# EE4410 Cryptography Practicum – Lab 2 Assignment
This notebook solves all tasks related to symmetric encryption (AES), asymmetric encryption (RSA), and digital signatures (RSA-PSS and ECDSA).

## 🔐 Symmetric Encryption (AES)

In [ ]:
# Install required libraries
!pip install pycryptodome requests

In [ ]:

from Crypto.Cipher import AES
from Crypto.Protocol.KDF import PBKDF2
from Crypto.Random import get_random_bytes
from base64 import b64encode, b64decode
import json, requests

def pad(data): return data + b' ' * (16 - len(data) % 16)
def unpad(data): return data.rstrip(b' ')

def get_key(passphrase, key_len):
    return PBKDF2(passphrase, b'salt', dkLen=key_len//8)

def encrypt(message, passphrase, key_len=128, mode=AES.MODE_CBC):
    key = get_key(passphrase, key_len)
    iv = get_random_bytes(16)
    cipher = AES.new(key, mode, iv=iv)
    ciphertext = cipher.encrypt(pad(message.encode()))
    return {'iv': b64encode(iv).decode(), 'ciphertext': b64encode(ciphertext).decode(), 'mode': mode}

def decrypt(enc_dict, passphrase, key_len=128):
    key = get_key(passphrase, key_len)
    iv = b64decode(enc_dict['iv'])
    ciphertext = b64decode(enc_dict['ciphertext'])
    cipher = AES.new(key, enc_dict['mode'], iv=iv)
    return unpad(cipher.decrypt(ciphertext)).decode()


In [ ]:

passphrase = "sharedsecret123"
message = "This is a test message."
aes_result = encrypt(message, passphrase, 128, AES.MODE_CBC)
print("Encrypted (AES-CBC):", aes_result)
print("Decrypted:", decrypt(aes_result, passphrase, 128))


In [ ]:

print("AES-GCM encrypted outputs:")
for i in range(3):
    gcm_result = encrypt("Same message", passphrase, 128, AES.MODE_GCM)
    print(f"Run {i+1}: {gcm_result['ciphertext']}")


## 🔑 Asymmetric Encryption (RSA)

In [ ]:

from Crypto.PublicKey import RSA
from Crypto.Cipher import PKCS1_OAEP

def generate_rsa_keys(bits):
    key = RSA.generate(bits)
    private_key = key.export_key()
    public_key = key.publickey().export_key()
    return private_key, public_key

def encrypt_rsa(message, public_key_bytes):
    pubkey = RSA.import_key(public_key_bytes)
    cipher_rsa = PKCS1_OAEP.new(pubkey)
    return b64encode(cipher_rsa.encrypt(message.encode())).decode()

def decrypt_rsa(ciphertext_b64, private_key_bytes):
    privkey = RSA.import_key(private_key_bytes)
    cipher_rsa = PKCS1_OAEP.new(privkey)
    return cipher_rsa.decrypt(b64decode(ciphertext_b64)).decode()

private_key, public_key = generate_rsa_keys(2048)
rsa_cipher = encrypt_rsa("RSA secret", public_key)
rsa_plain = decrypt_rsa(rsa_cipher, private_key)

print("Encrypted (RSA):", rsa_cipher)
print("Decrypted:", rsa_plain)


## ✍️ Digital Signatures (RSA-PSS and ECDSA)

In [ ]:

from Crypto.Signature import pss
from Crypto.Hash import SHA256
from Crypto.Signature import DSS
from Crypto.PublicKey import ECC

# RSA-PSS signature
rsa_key = RSA.import_key(private_key)
message = b"Message to sign"
h = SHA256.new(message)
signature = pss.new(rsa_key).sign(h)

# Verify signature
verifier = pss.new(RSA.import_key(public_key))
try:
    verifier.verify(h, signature)
    print("RSA-PSS signature is valid.")
except (ValueError, TypeError):
    print("Invalid RSA-PSS signature.")


In [ ]:

ecc_key = ECC.generate(curve='P-256')
h = SHA256.new(message)
signer = DSS.new(ecc_key, 'fips-186-3')
ecc_signature = signer.sign(h)

verifier = DSS.new(ecc_key.public_key(), 'fips-186-3')
try:
    verifier.verify(h, ecc_signature)
    print("ECDSA signature is valid.")
except (ValueError, TypeError):
    print("Invalid ECDSA signature.")


## 📝 Final Report Summary
(TODO: Fill in discussion points after executing cells)